In [ ]:
# 1. Install necessary libraries (takes ~2 mins)
!pip install -q transformers trl accelerate peft bitsandbytes flash-attn

In [ ]:
# 2. Imports
import os
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from google.colab import drive, runtime

In [ ]:
# 3. Mount Google Drive and create backup folder
drive.mount('/content/drive')

drive_path = '/content/drive/MyDrive/BitNet_CMDB_Model'
os.makedirs(drive_path, exist_ok=True)
print(f'Backup path ready: {drive_path}')

In [ ]:
# 4. Configuration
model_id = 'microsoft/bitnet-b1.58-3B-v1'  # Or the specific 1.58b variant
dataset_file = 'synthetic_cmdb_data.jsonl'
output_dir = './bitnet-cmdb-final'

In [ ]:
# 5. Load dataset, model, and tokenizer
print('Loading CMDB Dataset...')
dataset = load_dataset('json', data_files=dataset_file, split='train')

print('Loading BitNet Model (this may take a few minutes)...')
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# Using bfloat16 for training stability on L4/A100 GPUs
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

In [ ]:
# 6. Train and save model
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    num_train_epochs=3,
    logging_steps=10,
    save_strategy='no',  # Don't waste space/time saving intermediate checkpoints
    bf16=True,
    push_to_hub=False,
    report_to='none',  # Disable wandb/others to stay focused
)

# Newer TRL versions infer text from a 'text' field instead of dataset_text_field in __init__.
train_dataset = dataset.rename_column('output', 'text') if 'output' in dataset.column_names else dataset

trainer_kwargs = {
    'model': model,
    'args': training_args,
    'train_dataset': train_dataset,
}

try:
    trainer = SFTTrainer(processing_class=tokenizer, **trainer_kwargs)
except TypeError:
    # Backward compatibility for older TRL versions.
    trainer = SFTTrainer(tokenizer=tokenizer, **trainer_kwargs)

print('Starting Training Run...')
trainer.train()
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

In [ ]:
# 7. Archive and copy model to Google Drive
print('Compressing and moving model to Google Drive...')
!tar -czvf bitnet_cmdb_model.tar.gz {output_dir}
!cp bitnet_cmdb_model.tar.gz {drive_path}
print(f'Model successfully backed up to: {drive_path}')

In [ ]:
# 8. Optional: auto-disconnect runtime to save Colab units
print('Shutting down runtime to save compute units...')
runtime.unassign()